# HyperLux Minimal

Render **3 ProcTHOR scenes × 1,024 cameras** at **64×64**. Each scene has its own renderer; all share one compiled configuration. The first run downloads scenes and compiles native code.

With the OptiX backend, camera inputs and rendered frames stay on CUDA. Other backends use CPU tensors.

In [ ]:
%pip install -q hyperlux torch matplotlib

In [ ]:
import torch
import hyperlux
from hyperlux.camera import RGB

SCENES = (0, 1, 2, 3)
BATCH_SIZE, SIZE = 1024, 512
config = hyperlux.Config(width=SIZE, height=SIZE, batch_size=BATCH_SIZE, fidelity="veryhigh")
dataset = hyperlux.datasets.procthor.Train200()
renderers = []
for scene_id in SCENES:
    renderer = hyperlux.HyperLux(config=config)
    renderer.init(hyperlux.load_scene(dataset[scene_id].reference, fidelity=config.fidelity))
    renderers.append(renderer)

device = torch.device("cuda" if renderers[0].backend == "optix" else "cpu")
stream = torch.cuda.current_stream(device).cuda_stream if device.type == "cuda" else 0

Supply camera poses as float32 `(batch, 4, 3)` ray bases: **position, top-left ray, horizontal increment, vertical increment**, in world coordinates (+X forward, +Y left, +Z up). This example looks around each scene's center with a 90° field of view; replace the positions and headings with your own.

In [ ]:
yaw = torch.arange(BATCH_SIZE, device=device) * (2 * torch.pi / BATCH_SIZE)
forward = torch.stack((yaw.cos(), yaw.sin(), torch.zeros_like(yaw)), dim=1)
right = torch.stack((yaw.sin(), -yaw.cos(), torch.zeros_like(yaw)), dim=1)
up = torch.tensor([0., 0., 1.], device=device).expand_as(forward)
camera_bases = []
for renderer in renderers:
    positions = torch.tensor(renderer.scene_bounds["center"], dtype=torch.float32, device=device).expand_as(forward)
    camera_bases.append(torch.stack((positions, forward - right + up, 2 * right, -2 * up), dim=1))

Render RGBA frames into a `(scenes, cameras, height, width, 4)` tensor. Rerun this cell after updating `camera_bases`.

In [ ]:
frames = torch.empty((len(renderers), BATCH_SIZE, SIZE, SIZE, 4), dtype=torch.uint8, device=device)
for i, (renderer, cameras) in enumerate(zip(renderers, camera_bases)):
    renderer.set_cameras(cameras, stream=stream)
    renderer.render()
    renderer.outputs[RGB()].read(out=frames[i], stream=stream)
print(f"{frames.shape}: {frames.dtype} on {frames.device}")

Preview 64 views per scene in four rows of sixteen; only these previews are copied to the CPU.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("jpeg", bbox_inches=None, pil_kwargs={"quality": 80, "optimize": True})

preview = frames[:, ::BATCH_SIZE // 64, :, :, :3].cpu()
fig, axes = plt.subplots(4 * len(renderers), 16, figsize=(16, 4 * len(renderers)), dpi=100, squeeze=False)
for ax, image in zip(axes.flat, preview.flatten(0, 1)):
    ax.imshow(image)
    ax.axis("off")
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0.02, hspace=0.02)